# Preprocess `agentic_usage.csv` into `agentic_usage_swh.csv`

This notebook enriches `input/agentic_usage.csv` with an additional column containing the corresponding Software Heritage origin URL.

Important note: some GitHub origins are archived in SWH with a different case. We therefore compare URLs case-insensitively.

The SWH dump used here is `s3://softwareheritage/graph/2026-06-04/orc/origin/`. The ORC files are downloaded locally, then queried with DuckDB to perform the case-insensitive join.

In [1]:
from pathlib import Path
import shutil
import subprocess

import duckdb
import pandas as pd
import pyarrow.orc as orc

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE_DIR = Path("..")
INPUT_PATH = BASE_DIR / "input" / "agentic_usage.csv"
OUTPUT_PATH = BASE_DIR / "input" / "agentic_usage_swh.csv"
ORC_S3_PREFIX = "s3://softwareheritage/graph/2026-06-04/orc/origin/"
ORC_TMP_DIR = Path("data/orc_tmp")
ORC_LOCAL_DIR = ORC_TMP_DIR / "origin"
ORC_TMP_DIR.mkdir(parents=True, exist_ok=True)
ORC_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

MIN_FREE_BYTES = 2 * 1024**3

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Scratch dir: {ORC_LOCAL_DIR.resolve()}")

Input: ../input/agentic_usage.csv
Output: ../input/agentic_usage_swh.csv
Scratch dir: /home/rlefeuvr/agentic-ai-quality-diversity-reproduction/python/data/orc_tmp/origin


## Load input URLs

We keep the original rows and add one column: `swh_origin_url`. The output stays line-for-line compatible with the input CSV.

In [2]:
usage_df = pd.read_csv(INPUT_PATH)
usage_df["github_url_lower"] = usage_df["github_url"].str.lower()

targets = usage_df[["github_url", "github_url_lower"]].drop_duplicates().reset_index(drop=True)

print(f"{len(usage_df):,} input row(s)")
print(f"{len(targets):,} distinct GitHub URL(s)")

27,192 input row(s)
27,192 distinct GitHub URL(s)


## Download the ORC files locally

We first materialize the SWH export on disk. DuckDB will then read the local ORC files directly.

In [7]:
if not any(ORC_LOCAL_DIR.glob("*.orc")):
    free_bytes = shutil.disk_usage(ORC_TMP_DIR).free
    if free_bytes < MIN_FREE_BYTES:
        raise RuntimeError(
            f"Only {free_bytes / 1024**3:.2f} GB free in {ORC_TMP_DIR}, aborting before downloading the SWH export"
        )

    subprocess.run(
        [
            "aws", "s3", "sync", "--no-sign-request", "--only-show-errors",
            "--exclude", "*", "--include", "*.orc",
            ORC_S3_PREFIX,
            str(ORC_LOCAL_DIR),
        ],
        check=True,
    )
else:
    print(f"Reusing existing ORC files in {ORC_LOCAL_DIR}")

orc_count = sum(1 for _ in ORC_LOCAL_DIR.glob("*.orc"))
print(f"{orc_count} ORC file(s) available locally")

128 ORC file(s) available locally


## Load everything into DuckDB and match there

We stage the local ORC export into one DuckDB table, then let DuckDB handle the case-insensitive join and duplicate diagnostics in SQL.

In [5]:
con = duckdb.connect(str(ORC_TMP_DIR / "agentic_usage_swh.duckdb"))
con.execute("PRAGMA threads=4")
con.execute("DROP TABLE IF EXISTS origins")
con.execute("DROP TABLE IF EXISTS targets")
con.execute("DROP TABLE IF EXISTS matched_urls")
con.execute("DROP TABLE IF EXISTS duplicate_matches")

con.register("targets_df", targets)
con.execute("CREATE TEMP TABLE targets AS SELECT * FROM targets_df")

orc_paths = sorted(ORC_LOCAL_DIR.glob("*.orc"))
con.execute("CREATE TEMP TABLE origins(url VARCHAR)")

for orc_path in orc_paths:
    chunk = orc.ORCFile(str(orc_path)).read(columns=["url"]).to_pandas()
    con.register("orc_chunk_df", chunk)
    con.execute("INSERT INTO origins SELECT url FROM orc_chunk_df")

con.execute("ALTER TABLE origins ADD COLUMN url_lower VARCHAR")
con.execute("UPDATE origins SET url_lower = lower(url)")

# Find lowercase URLs in targets that map to multiple distinct-case origins
con.execute(
    """
    CREATE TEMP TABLE duplicate_matches AS
    SELECT
        t.github_url_lower,
        t.github_url,
        count(*) AS raw_match_count,
        count(DISTINCT o.url) AS distinct_match_count,
        list(DISTINCT o.url) AS candidate_urls
    FROM targets t
    JOIN origins o
      ON o.url_lower = t.github_url_lower
    GROUP BY t.github_url_lower, t.github_url
    HAVING count(DISTINCT o.url) > 1
    ORDER BY distinct_match_count DESC, t.github_url_lower
    """
)

duplicate_rows = con.execute("SELECT * FROM duplicate_matches").fetchdf()

# For duplicates: if the original-case URL matches one candidate exactly,
# keep that one and just print the others. Otherwise fall back to min().
exact_match_overrides = {}
if not duplicate_rows.empty:
    for row in duplicate_rows.itertuples(index=False):
        candidates = list(row.candidate_urls)
        exact_match = row.github_url if row.github_url in candidates else None

        if exact_match:
            others = [u for u in candidates if u != exact_match]
            print(
                f"Multiple SWH matches for {row.github_url_lower}: "
                f"{int(row.distinct_match_count)} distinct URLs -> exact match found, "
                f"keeping {exact_match}; other candidates: {others}"
            )
            exact_match_overrides[row.github_url_lower] = exact_match
        else:
            chosen = min(candidates)
            others = [u for u in candidates if u != chosen]
            print(
                f"Multiple SWH matches for {row.github_url_lower}: "
                f"{int(row.distinct_match_count)} distinct URLs, no exact case match; "
                f"keeping {chosen}; other candidates: {others}"
            )

# Base match: one row per lowercase URL, picking min(url) as default
con.execute(
    """
    CREATE TEMP TABLE matched_urls AS
    SELECT
        t.github_url_lower,
        min(o.url) AS swh_origin_url
    FROM targets t
    JOIN origins o
      ON o.url_lower = t.github_url_lower
    GROUP BY t.github_url_lower
    """
)

matches_df = con.execute("SELECT * FROM matched_urls").fetchdf()
match_map = dict(zip(matches_df["github_url_lower"], matches_df["swh_origin_url"]))

# Override with exact case-sensitive matches wherever one was found
match_map.update(exact_match_overrides)

usage_df["swh_origin_url"] = usage_df["github_url_lower"].map(match_map)
output_df = usage_df.drop(columns=["github_url_lower"])

output_df.to_csv(OUTPUT_PATH, index=False)

matched_rows = output_df["swh_origin_url"].notna().sum()
print(f"Wrote {len(output_df):,} row(s) to {OUTPUT_PATH}")
print(f"Matched rows: {matched_rows:,}")
print(f"Unmatched rows: {len(output_df) - matched_rows:,}")
output_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Multiple SWH matches for https://github.com/antvis/f2: 3 distinct URLs -> exact match found, keeping https://github.com/antvis/f2; other candidates: ['https://github.com/AntVis/f2', 'https://github.com/antvis/F2']
Multiple SWH matches for https://github.com/antvis/g2: 3 distinct URLs -> exact match found, keeping https://github.com/antvis/g2; other candidates: ['https://github.com/antvis/G2', 'https://github.com/AntVis/G2']
Multiple SWH matches for https://github.com/bxnxm/micros: 3 distinct URLs -> exact match found, keeping https://github.com/bxnxm/micros; other candidates: ['https://github.com/BxNxM/micrOS', 'https://github.com/BxNxM/MicrOs']
Multiple SWH matches for https://github.com/ecogenomics/gtdbtk: 3 distinct URLs -> exact match found, keeping https://github.com/ecogenomics/gtdbtk; other candidates: ['https://github.com/Ecogenomics/GTDBTk', 'https://github.com/Ecogenomics/GtdbTk']
Multiple SWH matches for https://github.com/freecodecamp/freecodecamp: 3 distinct URLs, no exact

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 27,192 row(s) to ../input/agentic_usag_swh.csv
Matched rows: 14,092
Unmatched rows: 13,100


,github_url,adoption_date,swh_origin_url
0,https://github.com/Kamyil/.dotfiles,2025-08-31T13:11:05+00:00,NaN
1,https://github.com/fatbobman/Objects2XLSX,2025-06-14T23:24:43+00:00,NaN
2,https://github.com/razorrun/react-native-vlc-m...,2025-08-18T18:53:37+00:00,NaN
3,https://github.com/funnyguilds/funnyguilds,2025-11-22T13:24:49+00:00,https://github.com/FunnyGuilds/FunnyGuilds
4,https://github.com/agustinusnathaniel/nextarte...,2025-12-31T06:53:35+00:00,NaN


In [6]:
output_df

,github_url,adoption_date,swh_origin_url
0,https://github.com/Kamyil/.dotfiles,2025-08-31T13:11:05+00:00,NaN
1,https://github.com/fatbobman/Objects2XLSX,2025-06-14T23:24:43+00:00,NaN
2,https://github.com/razorrun/react-native-vlc-m...,2025-08-18T18:53:37+00:00,NaN
3,https://github.com/funnyguilds/funnyguilds,2025-11-22T13:24:49+00:00,https://github.com/FunnyGuilds/FunnyGuilds
4,https://github.com/agustinusnathaniel/nextarte...,2025-12-31T06:53:35+00:00,NaN
...,...,...,...
27187,https://github.com/mczub/wtfdig,2025-07-02T05:05:09+00:00,https://github.com/mczub/wtfdig
27188,https://github.com/forus-labs/forui,2025-03-05T01:49:39+00:00,NaN
27189,https://github.com/reown-com/appkit,2025-03-14T15:47:52+00:00,NaN
27190,https://github.com/adamritter/pageql,2025-04-17T16:39:49+00:00,NaN


## Repos without `swh_origin_url`: GitHub creation date

For a random sample of 400 GitHub repos not matched in SWH, we query the GitHub API (`GET /repos/{owner}/{repo}`) to retrieve `created_at` and check whether the repo has existed since before 2021-01-01. Requires a GitHub token in the `GITHUB_TOKEN` (or `GH_TOKEN`) environment variable to benefit from the 5000 requests/hour rate limit.

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse

import requests
from tqdm.auto import tqdm

if "output_df" not in globals():
    output_df = pd.read_csv(Path("..") / "input" / "agentic_usage_swh.csv")
    print(f"Loaded output_df from disk: {len(output_df):,} row(s)")

GITHUB_TOKEN = ""
if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN (or GH_TOKEN) in the environment before running this cell")

GITHUB_API_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

CUTOFF_DATE = pd.Timestamp("2021-01-01", tz="UTC")
SAMPLE_SIZE = 1000
RANDOM_SEED = 42
RESULT_PATH = Path("../output-final") / "unmatched_created_before_2021_sample.csv"

unmatched = output_df.loc[output_df["swh_origin_url"].isna(), "github_url"].drop_duplicates()
print(f"{len(unmatched):,} distinct unmatched GitHub URL(s)")

sample_urls = unmatched.sample(n=min(SAMPLE_SIZE, len(unmatched)), random_state=RANDOM_SEED).tolist()
print(f"Sampled {len(sample_urls)} URL(s)")


def parse_owner_repo(github_url: str):
    path = urlparse(github_url).path.strip("/")
    parts = path.split("/")
    if len(parts) < 2:
        return None
    owner, repo = parts[0], parts[1]
    if repo.endswith(".git"):
        repo = repo[: -len(".git")]
    return owner, repo


def fetch_created_at(owner: str, repo: str):
    url = f"https://api.github.com/repos/{owner}/{repo}"
    resp = requests.get(url, headers=GITHUB_API_HEADERS, timeout=15)

    if resp.status_code == 200:
        return resp.json().get("created_at"), "ok"

    if resp.status_code == 404:
        return None, "not_found"

    if resp.status_code == 403 and resp.headers.get("X-RateLimit-Remaining") == "0":
        reset_at = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
        wait_s = max(reset_at - time.time(), 1)
        print(f"Rate limited, sleeping {wait_s:.0f}s")
        time.sleep(wait_s)
        return fetch_created_at(owner, repo)

    return None, f"error_{resp.status_code}"


results = []
for github_url in tqdm(sample_urls):
    parsed = parse_owner_repo(github_url)
    if parsed is None:
        results.append({"github_url": github_url, "created_at": None, "status": "bad_url"})
        continue

    owner, repo = parsed
    created_at, status = fetch_created_at(owner, repo)
    results.append({"github_url": github_url, "created_at": created_at, "status": status})

creation_sample_df = pd.DataFrame(results)
creation_sample_df["created_at"] = pd.to_datetime(creation_sample_df["created_at"], utc=True, errors="coerce")
creation_sample_df["created_before_2021"] = creation_sample_df["created_at"] < CUTOFF_DATE

RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
creation_sample_df.to_csv(RESULT_PATH, index=False)
print(f"Wrote {len(creation_sample_df):,} row(s) to {RESULT_PATH}")

print(creation_sample_df["status"].value_counts())
print()
ok_mask = creation_sample_df["status"] == "ok"
n_ok = int(ok_mask.sum())
n_before = int(creation_sample_df.loc[ok_mask, "created_before_2021"].sum())
print(f"{n_ok}/{len(creation_sample_df)} repos resolved via API")
if n_ok:
    print(f"{n_before}/{n_ok} ({n_before / n_ok:.1%}) resolved repos were created before 2021-01-01")

creation_sample_df

13,100 distinct unmatched GitHub URL(s)
Sampled 1000 URL(s)


  0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 1,000 row(s) to result/unmatched_created_before_2021_sample.csv
status
ok           983
not_found     17
Name: count, dtype: int64

983/1000 repos resolved via API
126/983 (12.8%) resolved repos were created before 2021-01-01


,github_url,created_at,status,created_before_2021
0,https://github.com/Enterprise-CMCS/managed-car...,2020-07-27 19:02:24+00:00,ok,True
1,https://github.com/cognitedata/opcua-extractor...,NaT,not_found,False
2,https://github.com/micake/micake,2019-11-01 10:11:24+00:00,ok,True
3,https://github.com/netidx/netidx,2018-05-21 21:36:51+00:00,ok,True
4,https://github.com/ChuckBuilds/LEDMatrix,2025-04-07 21:52:36+00:00,ok,False
...,...,...,...,...
995,https://github.com/DigitecGalaxus/next-yak,2023-06-09 09:31:13+00:00,ok,False
996,https://github.com/chainloop-dev/chainloop,2023-03-06 14:30:50+00:00,ok,False
997,https://github.com/sibiraj-s/eslint-plugin-fil...,2020-12-13 14:21:30+00:00,ok,True
998,https://github.com/txbody-org/sutra-cardano,2024-07-30 03:33:33+00:00,ok,False
